In [1]:
import sympy as sp

from sympy.abc import x, y, z, gamma

import numpy as np
from scipy.optimize import fsolve
from sympy import lambdify
from sympy.solvers import solve
from sympy.plotting import plot

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
#sigma gamma for a single shot taken at a time x, where T=1 (total time), and t_wait=1 (wait/reset time)
T, t_reset,d, sigma, n = sp.symbols("T t_{reset} \partial \sigma N")

d_gamma= sp.Eq(d *gamma,2/sp.sqrt(T)*sp.sqrt((x+t_reset)/(x*x))*sp.sqrt(sp.exp(gamma*x)-1))
d_gamma_2= sp.Eq(d *gamma,2*sp.sqrt((x+1)/(x*x))*sp.sqrt(sp.exp(gamma*x)-1))

In [3]:
d_gamma

Eq(\partial*gamma, 2*sqrt((t_{reset} + x)/x**2)*sqrt(exp(gamma*x) - 1)/sqrt(T))

In [5]:
sp.diff(sp.sqrt((x+t_reset)/(x*x*T))*sp.sqrt(sp.exp(gamma*x)-1),x)

T*x**2*sqrt((t_{reset} + x)/(T*x**2))*(1/(2*T*x**2) - (t_{reset} + x)/(T*x**3))*sqrt(exp(gamma*x) - 1)/(t_{reset} + x) + gamma*sqrt((t_{reset} + x)/(T*x**2))*exp(gamma*x)/(2*sqrt(exp(gamma*x) - 1))

In [161]:
sp.solve(sp.diff(2/sp.sqrt(T)*sp.sqrt((x+t_reset)/(x*x))*sp.sqrt(sp.exp(gamma*x)-1),x),x)

NotImplementedError: multiple generators [x, exp(gamma*x), sqrt(t_{reset}/x**2 + 1/x)]
No algorithms are implemented to solve equation gamma*sqrt((t_{reset} + x)/x**2)*exp(gamma*x)/(sqrt(T)*sqrt(exp(gamma*x) - 1)) + 2*x**2*sqrt((t_{reset} + x)/x**2)*(1/(2*x**2) - (t_{reset} + x)/x**3)*sqrt(exp(gamma*x) - 1)/(sqrt(T)*(t_{reset} + x))

In [132]:
d_gamma_2

Eq(\partial*gamma, 2*sqrt((x + 1)/x**2)*sqrt(exp(gamma*x) - 1))

In [133]:
#Differentiate sigma_d to find local minimum in respect to x (time)
sp.diff(2*sp.sqrt((x+1)/(x*x))*sp.sqrt(sp.exp(gamma*x)-1),x)

gamma*sqrt((x + 1)/x**2)*exp(gamma*x)/sqrt(exp(gamma*x) - 1) + 2*x**2*sqrt((x + 1)/x**2)*(1/(2*x**2) - (x + 1)/x**3)*sqrt(exp(gamma*x) - 1)/(x + 1)

In [134]:
#Trying to solve the equation = 0
sp.solve(sp.diff(2*sp.sqrt((x+1)/(x*x))*sp.sqrt(sp.exp(gamma*x)-1),x),x)

NotImplementedError: multiple generators [x, exp(gamma*x), sqrt(1/x + x**(-2))]
No algorithms are implemented to solve equation gamma*sqrt((x + 1)/x**2)*exp(gamma*x)/sqrt(exp(gamma*x) - 1) + 2*x**2*sqrt((x + 1)/x**2)*(1/(2*x**2) - (x + 1)/x**3)*sqrt(exp(gamma*x) - 1)/(x + 1)


from scipy.optimize import fsolve

# Define the equation as a function
def equation(x, gamma):
    term1 = gamma * np.sqrt((x + 1) / x**2) * np.exp(gamma * x) / np.sqrt(np.exp(gamma * x) - 1)
    term2 = 2 * x**2 * np.sqrt((x + 1) / x**2) * ((1 / (2 * x**2)) - (x + 1) / x**3) * np.sqrt(np.exp(gamma * x) - 1) / (x + 1)
    return term1 + term2

# Specify the value of gamma
gamma_value = np.linspace(0.1,10,100)

# Choose an initial guess for the solution
initial_guess = gamma_value/(np.arange(1,101))

# Solve the equation numerically
numerical_solution = fsolve(equation, initial_guess, args=(gamma_value,))

print("Numerical Solution:", numerical_solution)

from scipy.optimize import curve_fit

def fit_function(x, a, b):
    return a + b/x

# Fit the data using curve_fit
params, covariance = curve_fit(fit_function, gamma_value, numerical_solution)

# Extract the optimized parameters
a_opt, b_opt = params

# Plot original data
plt.scatter(gamma_value,numerical_solution, label='Original Data')

# Plot the fitted curve
x_fit = np.linspace(min(gamma_value), max(gamma_value), 100)
y_fit = fit_function(x_fit, a_opt, b_opt)
plt.plot(x_fit, y_fit, label='Fitted Curve', color='red')

plt.xlabel('gamma')
plt.ylabel('x_opt')
plt.legend()

### Plot sigma_T1 vs Total time in the case of mupltiple shots at a fixed time

#There's no analytical solution to the previous equation, so instead we have to use a numerical solver to find x_opt as a function of gamma

# Define the equation d_gamma=0 as a function
def equation(x, gamma):
    term1 = gamma * np.sqrt((x + 1) / x**2) * np.exp(gamma * x) / np.sqrt(np.exp(gamma * x) - 1)
    term2 = 2 * x**2 * np.sqrt((x + 1) / x**2) * ((1 / (2 * x**2)) - (x + 1) / x**3) * np.sqrt(np.exp(gamma * x) - 1) / (x + 1)
    return term1 + term2

# Specify the value of gamma (we use the theoretical value of gamma_relax)
gamma_value = np.linspace(0.1,10,1000)

# Choose an initial guess for the solution
initial_guess = 1.0/np.linspace(0.1,10,1000)


# Solve the equation numerically, i.e. x_opt(gammma) if gamma=gamma_relax
numerical_solution = fsolve(equation, initial_guess, args=(gamma_value,))

In [136]:
numerical_solution

array([0.2615949])

In [139]:
####### BREAK, NOW I WORK ON THE MULTIPLE SHOTS WITH DIFFERENT TIMES
f, g = sp.symbols("f g", cls = sp.Function)

sp.Eq(f(x), sp.exp(-gamma*x))

Eq(f(x), exp(-gamma*x))

In [144]:
sp.Eq(d*gamma, 2/(x*sp.sqrt(n))*sp.sqrt(sp.exp(gamma*x)-1))

Eq(\partial*gamma, 2*sqrt(exp(gamma*x) - 1)/(sqrt(N)*x))

In [145]:
sp.solve(sp.diff(2/x*sp.sqrt(sp.exp(gamma*x)-1),x),x)

[(LambertW(-2*exp(-2)) + 2)/gamma]

In [162]:
@@@@@@NEW for 3 poitns tryout

SyntaxError: invalid syntax (1116306593.py, line 1)